# Ozon LTV forecasting — project walkthrough

This lightweight notebook explains the reproducible pipeline. Heavy feature generation and model training live in importable modules and CLI scripts.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / 'src'))

from ozon_ltv.config import ANCHORS, VALIDATION_ANCHORS, TEST_ANCHOR
from ozon_ltv.datasets import eligible_train_anchors
from ozon_ltv.models import lightgbm_parameters

print(f'{len(ANCHORS)} train anchors: {ANCHORS[0]} .. {ANCHORS[-1]}')
print('Frozen validation:', VALIDATION_ANCHORS)
print('Test anchor:', TEST_ANCHOR)

## Leakage-safe split

A training target is admitted only when its full 30-day horizon ends before the validation anchor.

In [ ]:
{str(fold): [str(anchor) for anchor in eligible_train_anchors(fold)] for fold in VALIDATION_ANCHORS}

## Metric-aligned champion

RMSLE is optimized directly as RMSE over `log1p(target)`. The final model uses all 15 anchors and 965 trees.

In [ ]:
lightgbm_parameters(n_estimators=965, n_jobs=4)

## Confirmed results

| Experiment | RMSLE | Scope |
|---|---:|---|
| Previous-30-days baseline | 2.21618 | mean frozen CV |
| LightGBM | 1.73347 | mean frozen CV |
| LightGBM champion | **1.65408** | public leaderboard |
| Rank-feature variant | 1.66115 | public leaderboard, rejected |

Local CV and leaderboard values are shown separately to avoid overstating transfer.

## Reproduction commands

Run `pipelines/build_features.py`, then `pipelines/validate_lightgbm.py`, and finally `pipelines/train_champion.py`. Each pipeline accepts the source repository through `--project-root`; generated data is not duplicated here.